In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import timm
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time

batch_size = 32
# cifar_dataset_path = "./cifar10"
cifar_dataset_path = "~/LocalDev/datasets/cifar10"
transform = transforms.Compose([
    transforms.ToTensor()
])

# criando dataset de pastas
trainset = ImageFolder(root=f'{cifar_dataset_path}/train', transform=transform)
# criando dataloader (a partir de do dataset criado na linha anterior)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=0)

testset = ImageFolder(root=f'{cifar_dataset_path}/test', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=0)

# inspecionando as classes detectadas automaticamente (ordem alfabética das subpastas)
print("Classes detectadas pelo dataset de treino:", trainset.classes)
print("Observe o mapeamento da classe para índice:", trainset.class_to_idx)

# Nome das classes em ingles e portugues
# eu coloquei os nomes de clases retornados por trainset.classes
classes_en = ('airplane', 'automobile', 'bird', 'cat', 'deer', 
              'dog', 'frog', 'horse', 'ship', 'truck')
# e traduzi p/ portugues
classes_br = ('aviao', 'carro', 'ave', 'gato', 'cervo', 
              'cachorro', 'sapo', 'cavalo', 'navio', 'caminhao')


In [ ]:
# o nosso dataloader obtém imagens das 10 pastas de forma aleatória
# o codigo abaixo é para obter 10 imagens de cada classe
# para fins de visualizacao
dataiter = iter(trainloader)
class_samples = {x: [] for x in range(0, 10)}
incomplete = [x for x in range(0, 10)]
while incomplete:
    images, labels = next(dataiter)
    for img, lbl in zip(images, labels):
        lbl_folder = lbl.item()  # Índice da pasta (alfabético)
        if lbl_folder in incomplete:
            class_samples[lbl_folder].append(img.numpy().transpose(1, 2, 0))
            if len(class_samples[lbl_folder]) == 10:
                incomplete.remove(lbl_folder)


In [ ]:
# funcao para plotas imagens de cada classe
def plot_class_grid(class_samples, classes_br):
    fig, axes = plt.subplots(10, 10, figsize=(6, 6))
    for i, class_name in enumerate(classes_br):
        samples = class_samples[i]
        for j in range(10):
            ax = axes[i, j]
            ax.axis('off')
            img = samples[j]
            ax.imshow(img)
            if j == 0:
                ax.text(-10, img.shape[0]//2, class_name, 
                        va='center', ha='right', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_class_grid(class_samples, classes_br)

In [ ]:
# Carregando dataset em memória (vamos fazer isso para demonstrar o NN e KNN,
# aproveitando que o dataset é pequeno)

# vamos pegar todas imagens e carregar em numpy
def dataloader_para_numpy(dataloader):
    images_list = []
    labels_list = []
    print(f"Carregando dados do dataloader...")
    for batch_idx, (images, labels) in enumerate(dataloader):
        # No pytorch, as imagens do dataloader vem (batch_size, canais, altura, largura)
        # vamos converter pro formato -> (batch_size, altura, largura, canais)
        images_np = images.numpy().transpose(0, 2, 3, 1)
        labels_np = labels.numpy()
        images_list.append(images_np)
        labels_list.append(labels_np)
        if (batch_idx + 1) % 100 == 0:
            print(f"  Processados {(batch_idx + 1) * dataloader.batch_size} exemplos...")
    
    all_images = np.concatenate(images_list, axis=0)
    all_labels = np.concatenate(labels_list, axis=0)
    return all_images, all_labels

print("Carregando treino...")
X_train, y_train = dataloader_para_numpy(trainloader)
print(f"Dados de treino carregados: {X_train.shape}, {y_train.shape}")
print("Carregando teste...")
X_test, y_test = dataloader_para_numpy(testloader)
print(f"Dados de teste carregados: {X_test.shape}, {y_test.shape}")



In [ ]:
# implementação do Nearest Neighbour

class NearestNeighbour:
    def __init__(self):
        self.X_train = None
        self.y_train = None
    
    def train(self, X, y):
        # simplesmente armazena os dados de treino
        self.X_train = X
        self.y_train = y
    
    def predicao_nao_vetorizada(self, x, k=5, verbose=True):
        if verbose: print('\npredicao_nao_vetorizada')
        inicio = time.time()
        num_train = self.X_train.shape[0]
        distancias = np.zeros(num_train)
        # Loop não vetorizado
        for i in range(num_train):
            # Distância L1: soma de |x - x_train|
            distancias[i] = np.sum(np.abs(x - self.X_train[i]))
        
        tempo_gasto = time.time() - inicio
        
        if verbose: print("shape das distancias:", distancias.shape)
        indices = np.argsort(distancias)[:k]
        if verbose: print('indices mais próximos:', indices)
        if verbose: print('labels dos idx mais proximos:', [self.y_train[_i].item() for _i in indices])
        if verbose: print('scores dos indices:', distancias[indices])
        if verbose: print(f"Predicao NÃO vetorizada: {tempo_gasto:.4f} s")
        return indices, distancias[indices]
    
    def predicao_vetorizada(self, x, k=5, verbose=True):
        if verbose: print('\npredicao_vetorizada')
        # usa operações numpy vetorizadas
        inicio = time.time()
        # Operação vetorizada: calcula todas as distâncias de uma vez
        # Broadcasting: (N, H, W, C) - (H, W, C)
        distancias = np.sum(np.abs(self.X_train - x), axis=(1, 2, 3))

        tempo_gasto = time.time() - inicio
        
        if verbose: print("shape das distancias:", distancias.shape)
        indices = np.argsort(distancias)[:k]
        if verbose: print('indices mais próximos:', indices)
        if verbose: print('labels dos idx mais proximos:', [self.y_train[_i].item() for _i in indices])
        if verbose: print('scores dos indices:', distancias[indices])
        tempo_gasto = time.time() - inicio
        if verbose: print(f"Predicao VETORIZADA:     {tempo_gasto:.4f} s")
        return indices, distancias[indices]


In [ ]:
# "treinameto" - na verdade, simplesmente armazena o dataset de treino
nn = NearestNeighbour()
nn.train(X_train, y_train)

In [ ]:
# seleciona um item de teste aleatório
idx_imagem_teste = np.random.randint(0, len(X_test))
imagem_teste = X_test[idx_imagem_teste]
rotulo_teste = y_test[idx_imagem_teste]

print(f"\nImagem de teste - índice {idx_imagem_teste}")
print(f"shape da imagem: {imagem_teste.shape}")
print(f"Classe verdadeira: {classes_br[rotulo_teste]} ({classes_en[rotulo_teste]})")
print(f"Indice da classe verdadeira: {rotulo_teste}")

fig = plt.figure(figsize=(1,1))
plot = plt.imshow(imagem_teste)


In [ ]:
print(f"\nBuscando os 5 vizinhos mais próximos (nao vetorizado)\n")
indices_nv, distancias_nv = nn.predicao_nao_vetorizada(imagem_teste, k=5)


In [ ]:
print(f"\nBuscando os 5 vizinhos mais próximos (vetorizado)\n")
indices_v, distancias_v = nn.predicao_vetorizada(imagem_teste, k=5)

In [ ]:
# plot de 2 linhas e 3 colunas
fig, axes = plt.subplots(2, 5, figsize=(12, 2.5))

# linha 1
ax = axes[0, 0]
ax.imshow(imagem_teste)
ax.set_title(f"img teste: {classes_br[rotulo_teste]} ({rotulo_teste})", fontsize=9)
ax.axis('off')
for _i in range(1, 5): axes[0, _i].axis('off')
# linha 2
for i in range(5):
    ax = axes[1, i]
    vizinho_idx = indices_v[i]
    vizinho_imagem = X_train[vizinho_idx]
    vizinho_rotulo = y_train[vizinho_idx]
    distancia = distancias_v[i]
    
    ax.imshow(vizinho_imagem)
    ax.set_title(f"{i+1}: {classes_br[vizinho_rotulo]} - Dist L1: {distancia:.0f}", 
                 fontsize=8)
    ax.axis('off')


print(f"\nDistâncias dos 5 vizinhos mais próximos:")
for i, (idx, dist) in enumerate(zip(indices_v, distancias_v)):
    label = y_train[idx]
    print(f"  {i+1}. Classe: {classes_br[label]:12s} | Distância L1: {dist:10.2f}")

# Calcular acurácia simples (vizinho mais próximo)
predicted_label = y_train[indices_v[0]]
resultado = "Correto" if predicted_label == rotulo_teste else "Incorreto"
print(f"\nPredição (1-NN): {classes_br[predicted_label]} >>>> Resultado: {resultado}")

In [ ]:
# KNN
class KNN(NearestNeighbour):
    def __init__(self, k=5):
        super().__init__()
        self.k = k
    
    def predicao(self, x):
        k_indices, k_distances = super().predicao_vetorizada(x, k=self.k, verbose=False)
        # obtem labels dos k vizinhos mais próximos
        k_labels = self.y_train[k_indices]
        # obtem votacao majoritária
        contagem = np.bincount(k_labels)
        predicted_class = contagem.argmax()
        return predicted_class, k_labels

idx_imagem_teste = np.random.randint(0, len(X_test))
imagem_teste = X_test[idx_imagem_teste]
rotulo_teste = y_test[idx_imagem_teste]

knn = KNN(k=5)
knn.train(X_train, y_train)
classe_pred, suporte = knn.predicao(imagem_teste)
print(">>", classe_pred)
print(suporte)

In [ ]:
idx_imagem_teste = np.random.randint(0, len(X_test))
imagem_teste = X_test[idx_imagem_teste]
rotulo_teste = y_test[idx_imagem_teste]

knn = KNN(k=5)
knn.train(X_train, y_train)
classe_pred, suporte = knn.predicao(imagem_teste)
print(">>", classe_pred)
print(suporte)

In [ ]:
# Avaliação
# Criar e treinar o modelo
knn = KNN(k=5)
knn.train(X_train, y_train)

# Sortear 100 itens do dataset de teste
num_amostras = 100
num_acertos = num_erros = 0
indices_teste = np.random.choice(len(X_test), num_amostras, replace=False)

# Armazenar resultados
resultados = []
predicoes = []
rotulos_verdadeiros = []

print(f"\nAvaliando {num_amostras} amostras aleatórias do conjunto de teste...")
print("="*60)

for i, idx in enumerate(indices_teste):
    imagem_teste = X_test[idx]
    rotulo_teste = y_test[idx]
    # predição
    classe_pred, k_labels = knn.predicao(imagem_teste)
    predicoes.append(classe_pred)
    rotulos_verdadeiros.append(rotulo_teste)
    # acertou?
    acertou = (classe_pred == rotulo_teste)
    resultado = {
        'idx': idx,
        'rotulo_verdadeiro': rotulo_teste,
        'classe_predita': classe_pred,
        'k_labels': k_labels,
        'acertou': acertou,
        'imagem': imagem_teste
    }
    resultados.append(resultado)
    if acertou:
        num_acertos += 1
        status = "ACERTOU" 
    else:
        num_erros += 1
        status = "ERROU"
        
    print(f"Amostra {i+1:2d} (idx={idx:4d}): "
          f"Real={classes_br[rotulo_teste]:12s} | "
          f"Pred={classes_br[classe_pred]:12s} | "
          f"{status}")
    print(f"            Vizinhos (k={knn.k}): {[classes_br[lbl] for lbl in k_labels]}")

acuracia = 100*num_acertos/num_amostras
print("RESUMO FINAL")
print(f"Modelo: KNN com k={knn.k}")
print(f"Dataset de treino: {len(X_train)} imagens")
print(f"Amostras testadas: {num_amostras}")
print(f"Acurácia: {acuracia:.1f}%")
print(f"Acertos: {num_acertos}")
print(f"Erros: {num_erros}")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# matriz de confusao usando sklearn
matriz_confusao = confusion_matrix(rotulos_verdadeiros, predicoes)

print(f"Shape da matriz de conf.: {matriz_confusao.shape}")
print(f"Total de amostras: {np.sum(matriz_confusao)}")

# visualizacao

# subplot 1: Valores absolutos
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ax1 = axes[0]
im1 = ax1.imshow(matriz_confusao, cmap='Blues', aspect='auto')
# Adicionar valores nas células
for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_confusao[i, j]
        cor = 'white' if valor > matriz_confusao.max() / 2 else 'black'
        ax1.text(j, i, str(valor), ha='center', va='center', 
                color=cor, fontsize=10, fontweight='bold')

ax1.set_xticks(range(len(classes_br)))
ax1.set_yticks(range(len(classes_br)))
ax1.set_xticklabels(classes_br, rotation=45, ha='right')
ax1.set_yticklabels(classes_br)
ax1.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax1.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax1.set_title('Matriz de Confusão (Valores Absolutos)', fontsize=14, fontweight='bold')
cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('Número de Amostras', fontsize=10)

# subplot 2: Valores normalizados (%)
matriz_normalizada = matriz_confusao.astype('float') / matriz_confusao.sum(axis=1)[:, np.newaxis] * 100
ax2 = axes[1]
im2 = ax2.imshow(matriz_normalizada, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

# Adicionar valores nas células
for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_normalizada[i, j]
        cor = 'white' if valor > 50 else 'black'
        ax2.text(j, i, f'{valor:.1f}%', ha='center', va='center',
                color=cor, fontsize=9, fontweight='bold')

# Configurar eixos
ax2.set_xticks(range(len(classes_br)))
ax2.set_yticks(range(len(classes_br)))
ax2.set_xticklabels(classes_br, rotation=45, ha='right')
ax2.set_yticklabels(classes_br)
ax2.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax2.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax2.set_title('Matriz de Confusão Normalizada (%)', fontsize=14, fontweight='bold')

# Colorbar
cbar2 = plt.colorbar(im2, ax=ax2)
cbar2.set_label('Percentual (%)', fontsize=10)

plt.tight_layout()
plt.show()

# relatório do sklearn
report = classification_report(
    rotulos_verdadeiros, 
    predicoes, 
    target_names=classes_br,
    digits=3
)

print()
print(f"{report}")

print()
print("métricas")

acuracia_total = accuracy_score(rotulos_verdadeiros, predicoes) * 100
print(f"\nAcurácia Total: {acuracia_total:.2f}%")
print(f"Acertos: {np.sum(predicoes == rotulos_verdadeiros)}/{len(predicoes)}")

# acurácia por classe
acuracias = []
classes_presentes = []

for i in range(len(classes_br)):
    total_classe = np.sum(matriz_confusao[i, :])
    if total_classe > 0:
        acertos = matriz_confusao[i, i]
        acc = (acertos / total_classe) * 100
        acuracias.append(acc)
        classes_presentes.append(classes_br[i])
        print(f"{classes_br[i]:12s}: {acc:5.1f}% ({acertos}/{total_classe})")

# Plotar gráfico de barras
fig, ax = plt.subplots(figsize=(12, 6))

# Cores: verde se 100%, laranja se >= 50%, vermelho se < 50%
cores = []
for acc in acuracias:
    if acc == 100:
        cores.append('green')
    elif acc >= 50:
        cores.append('orange')
    else:
        cores.append('red')

bars = ax.bar(classes_presentes, acuracias, color=cores, alpha=0.7, edgecolor='black', linewidth=1.5)

# Adicionar valores nas barras
for bar, acc in zip(bars, acuracias):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{acc:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Linha da acurácia média
ax.axhline(y=acuracia_total, color='blue', linestyle='--', linewidth=2, 
           label=f'Acurácia Média: {acuracia_total:.1f}%')

ax.set_xlabel('Classe', fontsize=12, fontweight='bold')
ax.set_ylabel('Acurácia (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Acurácia por Classe (K={knn.k})', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='lower right')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

num_acertos = np.sum(np.array(predicoes) == np.array(rotulos_verdadeiros))
print(f"Modelo: KNN (k={knn.k})")
print(f"Amostras testadas: {len(predicoes)}")
print(f"Acurácia: {acuracia_total:.2f}%")
print(f"Acertos: {num_acertos}")
print(f"Erros: {len(predicoes) - num_acertos}")


## Treinamento com rede neural totalmente conectada (MLP)


#### Perceptron

<img src="images/MLP1.png" width=600>

#### Rede neural totalmente conectada

<img src="images/MLP2.png" width=600>

#### Grafo da uma rede neural

<img src="images/MLP3.png" width=600>



In [ ]:
# preparando o dataset

# precisamos dividir o dataset de treino (50.000) original em treino (90%) e validacao (10%)
# E criar os dataloaders

# Converter numpy arrays para tensors PyTorch
# X_train shape: (N, A, L, C) 
# Para MLP, vamos 'achatar': (N, A*L*C)

print(f"Shape original X_train: {X_train.shape}")
print(f"Shape original y_train: {y_train.shape}")

# Achatar as imagens para MLP
X_train_flat = X_train.reshape(X_train.shape[0], -1)  # (N, 32*32*3) = (N, 3072)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"Shape achatado X_train: {X_train_flat.shape}")

# Normalizar os dados para [0, 1] (já estão em [0, 1] se vieram do ToTensor)
# Se estão em [0, 255], dividir por 255
if X_train_flat.max() > 1.0:
    X_train_flat = X_train_flat / 255.0
    X_test_flat = X_test_flat / 255.0

# Converter para tensors
X_train_tensor = torch.FloatTensor(X_train_flat)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_flat)
y_test_tensor = torch.LongTensor(y_test)

print(f"\nTensor X_train: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
print(f"Tensor y_train: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}")
print(f"Range dos dados: [{X_train_tensor.min():.3f}, {X_train_tensor.max():.3f}]")

# particao do treino (50 k) vai ser dividida em treino e validacao

# criar dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# calcular tamanhos
total_size = len(train_dataset)
train_size = int(0.9 * total_size)
val_size = total_size - train_size

print(f"Total de amostras: {total_size}")
print(f"Treino (90%): {train_size} amostras")
print(f"Validação (10%): {val_size} amostras")

# dividir dataset
train_subset, val_subset = random_split(
    train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # para reprodutibilidade
)

# criar dataLoaders
batch_size = 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor), 
    batch_size=batch_size, 
    shuffle=False
)
print()
print(f"Batch size: {batch_size}")
print(f"Número de batches de treino: {len(train_loader)}")
print(f"Número de batches de validação: {len(val_loader)}")
print(f"Número de batches de teste: {len(test_loader)}")


In [ ]:
# arquitetura da rede
class MLP(nn.Module):
    def __init__(self, input_size=3072, hidden1_size=512, hidden2_size=256, num_classes=10):
        # rede 3 camadas (2 hidden + 1 output)
        super(MLP, self).__init__()
        # camada 1: entrada > h1
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        # camada 2: h1 > h2
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        # camada 3: h2 -> saida
        self.fc3 = nn.Linear(hidden2_size, num_classes)
        
    def forward(self, x):
        # camada 1
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        # camada 2
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        # camada 3 (output)
        x = self.fc3(x)
        
        return x


In [ ]:
# criando modelo na cpu
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print(f"Dispositivo: {device}")
model = MLP(input_size=3072, hidden1_size=512, hidden2_size=256, num_classes=10)
model = model.to(device)
print("Arquitetura do modelo:")
print(model)
# parâmetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parâmetros: {total_params}")
print(f"Parâmetros treináveis: {trainable_params}")

In [ ]:
# config. do treinamento
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"Funcao de perda (loss function): CrossEntropyLoss")
print(f"Otimizador: Adam (lr=0.001)")
print(f"Scheduler: ReduceLROnPlateau")


In [ ]:
# funcao de treino do modelo
def train_epoch(model, dataloader, criterion, optimizer, device):
    # treina o modelo por uma época (ou seja, no conjunto de treino)
    # o treinamento completo envolve várias épocas
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    # observem o tqdm
    for inputs, labels in tqdm(dataloader, desc="Treinando", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        #--------------- forward pass ---------------
        # passa os dados pelo modelo, gerando as predicoes
        outputs = model(inputs)
        # calcula a funcao de perda, comparando as predicoes do modelo com 
        # os rotulos verdadeiros (para ver o quanto errou)
        loss = criterion(outputs, labels)
        
        #--------------- backward pass ---------------
        # calcula os gradientes de perda em relação a todos parâmetros treinaveis
        loss.backward()
        # atualiza os pesos (parametros) usando os gradientes calculados 
        # e o algoritmo de otimização
        optimizer.step()
        
        #--------------- estatisticas ---------------
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# funcao de validacao
def validate(model, dataloader, criterion, device):
    # valida o modelo com os dados do dataloader
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validando", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            #--------------- forward pass ---------------
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            # NAO TEM backward pass (pq não estamos treinando)

            #--------------- estatisticas ---------------
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


In [ ]:
# treinamento
num_epochs = 50
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_state = None
print(f"iniciando treinamento por {num_epochs} épocas...\n")

for epoch in range(num_epochs):
    print(f"Época {epoch+1}/{num_epochs}")
    print("-" * 60)
    # treinar
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    # validar
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    # atualizar learning rate
    scheduler.step(val_loss)
    # salvar histórico
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    # salvar melhor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f">> Novo melhor modelo! Val Acc: {val_acc:.2f}%")
    # mostrar resultados
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print()


In [ ]:
# agora vamos observar como foi o treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# função de perda (loss function)
ax1 = axes[0]
ax1.plot(history['train_loss'], label='Treino', linewidth=2)
ax1.plot(history['val_loss'], label='Validação', linewidth=2)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss durante o Treinamento', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# acuracia
ax2 = axes[1]
ax2.plot(history['train_acc'], label='Treino', linewidth=2)
ax2.plot(history['val_acc'], label='Validação', linewidth=2)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Acurácia (%)', fontsize=12)
ax2.set_title('Acurácia durante o Treinamento', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# agora vamos testar com o melhor modelo
model.load_state_dict(best_model_state)
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Resultados no conjunto de teste:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.2f}%")


In [ ]:
# matriz de confusão

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Criar matriz de confusão
matriz_confusao_test = np.zeros((len(classes_br), len(classes_br)), dtype=int)
for i in range(len(all_labels)):
    matriz_confusao_test[all_labels[i], all_preds[i]] += 1

# visualizacao
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Subplot 1: Valores absolutos
ax1 = axes[0]
im1 = ax1.imshow(matriz_confusao_test, cmap='Blues', aspect='auto')

for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_confusao_test[i, j]
        # Cor do texto: branco se fundo escuro, preto se fundo claro
        cor = 'white' if valor > matriz_confusao_test.max() / 2 else 'black'
        ax1.text(j, i, str(valor), ha='center', va='center',
                color=cor, fontsize=9, fontweight='bold')

ax1.set_xticks(range(len(classes_br)))
ax1.set_yticks(range(len(classes_br)))
ax1.set_xticklabels(classes_br, rotation=45, ha='right')
ax1.set_yticklabels(classes_br)
ax1.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax1.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax1.set_title('Matriz de Confusão - MLP (Valores Absolutos)', fontsize=14, fontweight='bold')

cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('Número de Amostras', fontsize=10)

# Subplot 2: Valores normalizados (%)
matriz_normalizada_test = np.zeros_like(matriz_confusao_test, dtype=float)
for i in range(len(classes_br)):
    total_linha = np.sum(matriz_confusao_test[i, :])
    if total_linha > 0:
        matriz_normalizada_test[i, :] = (matriz_confusao_test[i, :] / total_linha) * 100

ax2 = axes[1]
im2 = ax2.imshow(matriz_normalizada_test, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_normalizada_test[i, j]
        # Cor do texto baseada no valor
        cor = 'white' if valor > 50 else 'black'
        ax2.text(j, i, f'{valor:.1f}%', ha='center', va='center',
                color=cor, fontsize=8, fontweight='bold')

ax2.set_xticks(range(len(classes_br)))
ax2.set_yticks(range(len(classes_br)))
ax2.set_xticklabels(classes_br, rotation=45, ha='right')
ax2.set_yticklabels(classes_br)
ax2.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax2.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax2.set_title('Matriz de Confusão - MLP (Normalizada %)', fontsize=14, fontweight='bold')

cbar2 = plt.colorbar(im2, ax=ax2)
cbar2.set_label('Percentual (%)', fontsize=10)

plt.tight_layout()
plt.show()

## Treinamento em uma rede neural convolucional (CNN)

#### Imagem RGB

<img src="images/ImageRGB.png" width=600>

#### Filtro de convolução

<img src="images/Filtro.png" width=300>

#### Convolução

<img src="images/Conv.png" width=600>

#### Mapa de ativação

<img src="images/Conv2.png" width=600>

#### Max Pooling

<img src="images/MaxPooling.png" width=600>

#### Exemplo rede VGG

<img src="images/VGG.png" width=600>


In [ ]:
# preparando o dataset

# precisamos dividir o dataset de treino (50.000) original em treino (90%) e validacao (10%)
# E criar os dataloaders
print(f"\nShape original X_train: {X_train.shape}")  # (N, 32, 32, 3)

# Transpor de (N, H, W, C) para (N, C, H, W)
X_train_cnn = np.transpose(X_train, (0, 3, 1, 2))  # (N, 3, 32, 32)
X_test_cnn = np.transpose(X_test, (0, 3, 1, 2))

print(f"Shape para CNN X_train: {X_train_cnn.shape}")  # (N, 3, 32, 32)

# Normalizar para [0, 1]
if X_train_cnn.max() > 1.0:
    X_train_cnn = X_train_cnn / 255.0
    X_test_cnn = X_test_cnn / 255.0

# Converter para tensors
X_train_tensor = torch.FloatTensor(X_train_cnn)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_cnn)
y_test_tensor = torch.LongTensor(y_test)

print(f"\nTensor X_train: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
print(f"Tensor y_train: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}")
print(f"Range dos dados: [{X_train_tensor.min():.3f}, {X_train_tensor.max():.3f}]")

# criar dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# calcular tamanhos
total_size = len(train_dataset)
train_size = int(0.9 * total_size)
val_size = total_size - train_size

print(f"\nTotal de amostras: {total_size}")
print(f"Treino (90%): {train_size} amostras")
print(f"Validação (10%): {val_size} amostras")

# dividir dataset
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# criar DataLoaders
batch_size = 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

print(f"\nBatch size: {batch_size}")
print(f"Número de batches de treino: {len(train_loader)}")
print(f"Número de batches de validação: {len(val_loader)}")
print(f"Número de batches de teste: {len(test_loader)}")

In [ ]:
# arquitetura da rede
class AlexNetCIFAR(nn.Module):
    def __init__(self, num_classes=10):
        """
        AlexNet adaptada para CIFAR-10 (imagens 32x32)
        
        A AlexNet original foi projetada para ImageNet (224x224).
        Esta versão é adaptada para imagens menores (32x32).
        
        Estrutura:
        - 5 camadas convolucionais
        - 3 camadas fully connected
        - ReLU, MaxPooling, Dropout
        """
        super(AlexNetCIFAR, self).__init__()
        
        # Camadas convolucionais (feature extractor)
        self.features = nn.Sequential(
            # Conv1: 3 -> 64 canais, kernel 3x3, stride 1, padding 1
            # Input: (3, 32, 32) -> Output: (64, 32, 32)
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (64, 16, 16)
            
            # Conv2: 64 -> 192 canais
            # Input: (64, 16, 16) -> Output: (192, 16, 16)
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (192, 8, 8)
            
            # Conv3: 192 -> 384 canais
            # Input: (192, 8, 8) -> Output: (384, 8, 8)
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv4: 384 -> 256 canais
            # Input: (384, 8, 8) -> Output: (256, 8, 8)
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv5: 256 -> 256 canais
            # Input: (256, 8, 8) -> Output: (256, 8, 8)
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (256, 4, 4)
        )
        
        # Adaptive pooling para garantir tamanho fixo
        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))  # (256, 4, 4)
        
        # Camadas fully connected (classifier)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 2048),  # 4096 pixels -> 2048
            nn.ReLU(inplace=True),
            
            nn.Dropout(0.5),
            nn.Linear(2048, 1024),  # 2048 -> 1024
            nn.ReLU(inplace=True),
            
            nn.Linear(1024, num_classes),  # 1024 -> 10 classes
        )
    
    def forward(self, x):
        """Forward pass"""
        # Extração de features
        x = self.features(x)
        
        # Adaptive pooling
        x = self.avgpool(x)
        
        # Achatar
        x = torch.flatten(x, 1)
        
        # Classificação
        x = self.classifier(x)
        
        return x

In [ ]:
# criando modelo na cpu
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print(f"Dispositivo: {device}")
model_cnn = AlexNetCIFAR(num_classes=10)
model_cnn = model_cnn.to(device)
print("Arquitetura do modelo:")
print(model_cnn)
# parâmetros
total_params = sum(p.numel() for p in model_cnn.parameters())
trainable_params = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")


In [ ]:
# config. do treinamento
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"\nLoss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr=0.001, weight_decay=1e-4)")
print(f"Scheduler: ReduceLROnPlateau (patience=5)")

In [ ]:
# funcao de treino do modelo
def train_epoch_cnn(model, dataloader, criterion, optimizer, device):
    # treina o modelo por uma época (ou seja, no conjunto de treino)
    # o treinamento completo envolve várias épocas
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in tqdm(dataloader, desc="Treinando", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # backward pass
        loss.backward()
        optimizer.step()
        
        # estatísticas
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


In [ ]:
# funcao de validacao
def validate_cnn(model, dataloader, criterion, device):
    # valida o modelo com os dados do dataloader
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validando", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # estatísticas
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# treinamento
num_epochs = 10
history_cnn = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_state = None

print(f"Iniciando treinamento por {num_epochs} épocas...\n")

for epoch in range(num_epochs):
    print(f"Época {epoch+1}/{num_epochs}")
    print("-" * 60)
    
    # Treinar
    train_loss, train_acc = train_epoch_cnn(model_cnn, train_loader, criterion, optimizer, device)
    
    # Validar
    val_loss, val_acc = validate_cnn(model_cnn, val_loader, criterion, device)
    
    # Atualizar learning rate
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    
    if new_lr != old_lr:
        print(f"Learning rate reduzido: {old_lr:.6f} -> {new_lr:.6f}")
    
    # Salvar histórico
    history_cnn['train_loss'].append(train_loss)
    history_cnn['train_acc'].append(train_acc)
    history_cnn['val_loss'].append(val_loss)
    history_cnn['val_acc'].append(val_acc)
    
    # Salvar melhor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model_cnn.state_dict().copy()
        print(f"Novo melhor modelo! Val Acc: {val_acc:.2f}%")
    
    # Mostrar resultados
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print()
    
    # Early stopping se já estiver muito bom
    if val_acc > 75:
        print(f"Acurácia alvo atingida! Parando treinamento.")
        break

print(f"Melhor acurácia de validação: {best_val_acc:.2f}%")


In [ ]:
# visualizar treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1 = axes[0]
ax1.plot(history_cnn['train_loss'], label='Treino', linewidth=2, marker='o', markersize=4)
ax1.plot(history_cnn['val_loss'], label='Validação', linewidth=2, marker='s', markersize=4)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss durante o Treinamento - AlexNet', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Acurácia
ax2 = axes[1]
ax2.plot(history_cnn['train_acc'], label='Treino', linewidth=2, marker='o', markersize=4)
ax2.plot(history_cnn['val_acc'], label='Validação', linewidth=2, marker='s', markersize=4)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Acurácia (%)', fontsize=12)
ax2.set_title('Acurácia durante o Treinamento - AlexNet', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# agora vamos testar com o melhor modelo
model_cnn.load_state_dict(best_model_state)
test_loss, test_acc = validate(model_cnn, test_loader, criterion, device)

print(f"Resultados no conjunto de teste:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.2f}%")

In [ ]:
# matriz de confusão
model_cnn.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model_cnn(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Criar matriz de confusão
matriz_confusao_cnn = np.zeros((len(classes_br), len(classes_br)), dtype=int)
for i in range(len(all_labels)):
    matriz_confusao_cnn[all_labels[i], all_preds[i]] += 1

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Valores absolutos
ax1 = axes[0]
im1 = ax1.imshow(matriz_confusao_cnn, cmap='Blues', aspect='auto')

for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_confusao_cnn[i, j]
        cor = 'white' if valor > matriz_confusao_cnn.max() / 2 else 'black'
        ax1.text(j, i, str(valor), ha='center', va='center',
                color=cor, fontsize=9, fontweight='bold')

ax1.set_xticks(range(len(classes_br)))
ax1.set_yticks(range(len(classes_br)))
ax1.set_xticklabels(classes_br, rotation=45, ha='right')
ax1.set_yticklabels(classes_br)
ax1.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax1.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax1.set_title('Matriz de Confusão - AlexNet (Valores Absolutos)', fontsize=14, fontweight='bold')

cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('Número de Amostras', fontsize=10)

# Normalizada
matriz_normalizada_cnn = np.zeros_like(matriz_confusao_cnn, dtype=float)
for i in range(len(classes_br)):
    total_linha = np.sum(matriz_confusao_cnn[i, :])
    if total_linha > 0:
        matriz_normalizada_cnn[i, :] = (matriz_confusao_cnn[i, :] / total_linha) * 100

ax2 = axes[1]
im2 = ax2.imshow(matriz_normalizada_cnn, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

for i in range(len(classes_br)):
    for j in range(len(classes_br)):
        valor = matriz_normalizada_cnn[i, j]
        cor = 'white' if valor > 50 else 'black'
        ax2.text(j, i, f'{valor:.1f}%', ha='center', va='center',
                color=cor, fontsize=8, fontweight='bold')

ax2.set_xticks(range(len(classes_br)))
ax2.set_yticks(range(len(classes_br)))
ax2.set_xticklabels(classes_br, rotation=45, ha='right')
ax2.set_yticklabels(classes_br)
ax2.set_xlabel('Classe Predita', fontsize=12, fontweight='bold')
ax2.set_ylabel('Classe Real', fontsize=12, fontweight='bold')
ax2.set_title('Matriz de Confusão - AlexNet (Normalizada %)', fontsize=14, fontweight='bold')

cbar2 = plt.colorbar(im2, ax=ax2)
cbar2.set_label('Percentual (%)', fontsize=10)

plt.tight_layout()
plt.show()

# acurácia por classe
print("Acurácia por Classe (Teste):")
print("-"*60)
for i in range(len(classes_br)):
    total_classe = np.sum(matriz_confusao_cnn[i, :])
    if total_classe > 0:
        acertos = matriz_confusao_cnn[i, i]
        acc = (acertos / total_classe) * 100
        print(f"{classes_br[i]:12s}: {acc:5.1f}% ({acertos}/{total_classe})")
        

## Fine-tuning de uma rede leve 


In [ ]:
# preparando dataset 

# Transpor de (N, A, L, C) para (N, A, L, W)
X_train_cnn = np.transpose(X_train, (0, 3, 1, 2))  # (N, 3, 32, 32)
X_test_cnn = np.transpose(X_test, (0, 3, 1, 2))

print(f"Shape para CNN X_train: {X_train_cnn.shape}")  # (N, 3, 32, 32)

# Normalizar para [0, 1]
if X_train_cnn.max() > 1.0:
    X_train_cnn = X_train_cnn / 255.0
    X_test_cnn = X_test_cnn / 255.0


# Normalizar com estatísticas do ImageNet
imagenet_mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
imagenet_std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

# Aplicar normalização
X_train_normalized = (X_train_cnn - imagenet_mean) / imagenet_std
# del X_train_cnn

X_test_normalized = (X_test_cnn - imagenet_mean) / imagenet_std
# del X_test_cnn

print(f"Range após normalização: [{X_train_normalized.min():.3f}, {X_train_normalized.max():.3f}]")

# converter para tensors
X_train_tensor = torch.FloatTensor(X_train_normalized)
del X_train_normalized

y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_normalized)
del X_test_normalized

y_test_tensor = torch.LongTensor(y_test)

print(f"\nTensor X_train: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")

# criar dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# calcular tamanhos
total_size = len(train_dataset)
train_size = int(0.9 * total_size)
val_size = total_size - train_size

print(f"\nTotal de amostras: {total_size}")
print(f"Treino (90%): {train_size} amostras")
print(f"Validação (10%): {val_size} amostras")

# dividir dataset
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
del train_dataset

# criar DataLoaders
batch_size = 64  # Batch menor para modelo maior
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

print(f"\nBatch size: {batch_size}")
print(f"Número de batches de treino: {len(train_loader)}")
print(f"Número de batches de validação: {len(val_loader)}")


In [ ]:
 preparando o dataset

# parte já feita na CNN
"""
# Transpor de (N, H, W, C) para (N, C, H, W)
X_train_cnn = np.transpose(X_train, (0, 3, 1, 2))  # (N, 3, 32, 32)
X_test_cnn = np.transpose(X_test, (0, 3, 1, 2))
print(f"Shape para CNN X_train: {X_train_cnn.shape}")  # (N, 3, 32, 32)
# Normalizar para [0, 1]
if X_train_cnn.max() > 1.0:
    X_train_cnn = X_train_cnn / 255.0
    X_test_cnn = X_test_cnn / 255.0
"""    

# conferindo normalizacao da CNN - esperamos nao encontrar um valor maior que 1.
print("X_train_cnn.max() antes de normalizar = ", X_train_cnn.max())

# Normalizar com estatísticas do ImageNet
imagenet_mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
imagenet_std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

# Aplicar normalização
X_train_normalized = (X_train_cnn - imagenet_mean) / imagenet_std
X_test_normalized = (X_test_cnn - imagenet_mean) / imagenet_std

print(f"Range após normalização: [{X_train_normalized.min():.3f}, {X_train_normalized.max():.3f}]")

# converter para tensors
X_train_tensor = torch.FloatTensor(X_train_normalized)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_normalized)
y_test_tensor = torch.LongTensor(y_test)

print(f"\nTensor X_train: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")

# criar dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# calcular tamanhos
total_size = len(train_dataset)
train_size = int(0.9 * total_size)
val_size = total_size - train_size

print(f"\nTotal de amostras: {total_size}")
print(f"Treino (90%): {train_size} amostras")
print(f"Validação (10%): {val_size} amostras")

# dividir dataset
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# criar DataLoaders
batch_size = 64  # Batch menor para modelo maior
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

print(f"\nBatch size: {batch_size}")
print(f"Número de batches de treino: {len(train_loader)}")
print(f"Número de batches de validação: {len(val_loader)}")

In [ ]:
# carregar modelo pre-trinado

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print(f"\nDispositivo: {device}")

# carregar modelo pré-treinado no ImageNet
print("\nCarregando rede pré-treinada...")
# model_efficient = timm.create_model('efficientnet_b0', pretrained=True, num_classes=10)
model_finetune = timm.create_model('mobilenetv3_small_100', pretrained=True, num_classes=10)

drop_rate=0.2,  # Dropout para regularização
# Mover para device
model_finetune = model_finetune.to(device)

print("\nModelo carregado com sucesso!")
print(f"Arquitetura: mobilenetv3_small_100")
print(f"Pré-treinamento: ImageNet")
print(f"Adaptado para: {10} classes (CIFAR-10)")

# Contar parâmetros
total_params = sum(p.numel() for p in model_finetune.parameters())
trainable_params = sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)

print(f"\nTotal de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

# Ver estrutura do classificador
print("\nEstrutura do classificador final:")
print(model_finetune.classifier)

#### Estratégia comuns de Fine-tuning

**Estratégia 1**: Treinar apenas o classificador (feature extraction)

- Congelar todas as camadas convolucionais

- Treinar apenas a camada final (classifier)

- Mais rápido, menos propenso a overfitting

**Estratégia 2**: Fine-tuning completo

- Descongelar todas as camadas

- Learning rate menor para camadas pré-treinadas

- Melhor performance, mas requer mais cuidado

**Estratégia 3**: fine-tuning em 2 fases

- Fase 1: Treinar apenas classificador (ex: 5 épocas)

- Fase 2: Fine-tuning completo com taxa de aprendizado (learning rate 'LR') baixo - (15 épocas)

In [ ]:
# fase 1 - Treinar apenas o classificador (feature extraction)

# congelar todas as camadas exceto o classificador
for name, param in model_finetune.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# verificar camadas congeladas
trainable_params_phase1 = sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params_phase1

print(f"\nParâmetros congelados: {frozen_params:,}")
print(f"Parâmetros treináveis: {trainable_params_phase1:,}")

In [ ]:
# config. do treinamento - fase 1
criterion = nn.CrossEntropyLoss()
optimizer_phase1 = optim.Adam(
    filter(lambda p: p.requires_grad, model_finetune.parameters()),
    lr=0.001
)

print(f"Otimizacao na fase 1: Adam (lr=0.001)")


In [ ]:
# Função de treino

def train_epoch_cnn(model, dataloader, criterion, optimizer, device, resize_to=224):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Treinando")
    
    # for inputs, labels in tqdm(dataloader, desc="Treinando", leave=False):
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)  # Agora inputs: (batch, 3, 224, 224)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        current_loss = running_loss / total
        current_acc = 100 * correct / total
        pbar.set_description(f"Treinando - Loss: {current_loss:.4f}, Acc: {current_acc:.2f}%")
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# Função de validacao

def validate_cnn(model, dataloader, criterion, device):
    # valida o modelo com os dados do dataloader
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validando", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # estatísticas
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# treinar fase 1
num_epochs_phase1 = 5
history_phase1 = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"Iniciando Fase 1: {num_epochs_phase1} épocas\n")

for epoch in range(num_epochs_phase1):
    print(f"Época {epoch+1}/{num_epochs_phase1}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch_cnn(model_finetune, train_loader, criterion, optimizer_phase1, device)
    val_loss, val_acc = validate_cnn(model_finetune, val_loader, criterion, device)
    
    history_phase1['train_loss'].append(train_loss)
    history_phase1['train_acc'].append(train_acc)
    history_phase1['val_loss'].append(val_loss)
    history_phase1['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print()

print("Fase 1 concluída!")

In [ ]:
# mais 5
num_epochs_phase1 = 10
for epoch in range(5, num_epochs_phase1):
    print(f"Época {epoch+1}/{num_epochs_phase1}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch_cnn(model_finetune, train_loader, criterion, optimizer_phase1, device)
    val_loss, val_acc = validate_cnn(model_finetune, val_loader, criterion, device)
    
    history_phase1['train_loss'].append(train_loss)
    history_phase1['train_acc'].append(train_acc)
    history_phase1['val_loss'].append(val_loss)
    history_phase1['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print()

print("Fase 1 concluída!")